# Guidelines for basic robot model generation

Basic robot MJCF files in `robotblockset/mujoco/mjcf_models` should use **local names without the `BaseName` prefix**. The provided models, such as `panda.xml`, `ur10e.xml`, and `iiwa14.xml`, follow this convention.

When a complete scene is generated, this tutorial adds the robot prefix automatically. Local names such as `joint1`, `TCP`, and `actuator1` then become names such as `<BaseName>_joint1`, `<BaseName>_TCP`, and `<BaseName>_actuator1` in the generated scene. The MuJoCo robot wrappers can then use those prefixed names consistently.

## 1. Main naming rule

- In the **standalone robot MJCF model**, do not use the `<BaseName>_` prefix.
- In the **generated scene**, the `<BaseName>_` prefix is added automatically.
- This rule applies to bodies, joints, sites, sensors, actuators, and all `default class` names used inside the robot model.

The standalone robot model should contain names such as:

- `base`, `link1`, `link7`
- `joint1`, `joint2`, `wrist_3_joint`
- `flange`, `TCP`, `FT`
- `pos`, `ori`, `v`, `w`, `force`, `torque`
- `robot`, `visual`, `collision`, `size4`, `joint1`

It should not contain names such as `Panda_joint1`, `Panda_TCP`, `Panda_pos`, or `Panda_robot`.

## 2. Base body and links

The root body in the standalone model should use a local name, usually `base`. Internal links should also use local names such as `link0`, `link1`, `base_link`, or `shoulder_link`.

Recommended:

```xml
<body name="base" pos="0 0 0">
    ...
</body>
```

After scene generation, this body becomes something like `Panda_base` or `UR10e_base`. In this tutorial, the generated base body is renamed to the robot instance name when needed for automatic wrapper handle generation.

## 3. Joint naming

Use local joint names in the standalone model:

- Generic models can use `joint1`, `joint2`, ..., `jointN`.
- Robot-specific names such as `shoulder_pan_joint` or `wrist_3_joint` are also valid.
- Do not add `<BaseName>_` inside the standalone model.
- If robot-specific joint names are used, the Python robot wrapper must know those names.

Joint names can be provided by defining `self.joint_names` in the corresponding robot specification in `robot_spec.py`, or by passing `JointNames=...` when creating the robot object.

Recommended generic names:

```xml
<joint name="joint1" .../>
<joint name="joint2" .../>
<joint name="joint3" .../>
```

Recommended robot-specific names:

```xml
<joint name="shoulder_pan_joint" .../>
<joint name="elbow_joint" .../>
<joint name="wrist_3_joint" .../>
```

When the scene is generated, these names are prefixed automatically, for example `Panda_joint1` or `UR10e_wrist_3_joint`.

This is especially important for robots such as UR10e, whose joints are not named `joint1`, `joint2`, and so on in the MJCF model. In that case, the same special names must be provided in the robot specification or through constructor arguments so the wrapper can map scene joints correctly.

## 4. Actuator naming

Actuator names in the standalone model should also be local and unprefixed. The default actuator names are sequential names such as `actuator1`, `actuator2`, ..., `actuatorN`.

The actuator type is determined by the MJCF element itself, for example `<position .../>` or `<general .../>`, not by the actuator name. If robot-specific actuator names are needed, keep them local and unprefixed inside the standalone model.

Recommended:

```xml
<position name="actuator1" joint="joint1" .../>
<position name="actuator2" joint="joint2" .../>
```

In the generated scene, these become `<BaseName>_actuator1`, `<BaseName>_actuator2`, and so on.

## 5. Sensor naming

Sensor names in the standalone model should also be unprefixed.

Recommended local names:

- Joint position sensors: `pos_joint1`, ..., `pos_jointN`
- Joint velocity sensors: `vel_joint1`, ..., `vel_jointN`
- Cartesian position sensor: `pos`
- Cartesian orientation sensor: `ori`
- Cartesian linear velocity sensor: `v`
- Cartesian angular velocity sensor: `w`
- Force sensor: `force`
- Torque sensor: `torque`

Typical sensor block:

```xml
<sensor>
    <jointpos    name="pos_joint1" joint="joint1"/>
    <jointvel    name="vel_joint1" joint="joint1"/>
    <framepos    name="pos" objtype="site" objname="TCP"/>
    <framequat   name="ori" objtype="site" objname="TCP"/>
    <framelinvel name="v" objtype="site" objname="TCP"/>
    <frameangvel name="w" objtype="site" objname="TCP"/>
    <force       name="force" site="FT"/>
    <torque      name="torque" site="FT"/>
</sensor>
```

After insertion into a scene, these names become `<BaseName>_pos`, `<BaseName>_ori`, `<BaseName>_force`, and so on.

## 6. Flange and TCP sites

Two sites are especially important in the standalone robot model:

- `flange`: the mechanical mounting interface of the bare robot arm.
- `TCP`: the tool center point used for task-space motion and Cartesian sensing.

The wrappers expect these sites as `<BaseName>_flange` and `<BaseName>_TCP` only after the scene generator has added the prefix. If both sites exist, the wrappers can compute the transform from flange to TCP automatically.

Recommended pattern inside the last robot link or tool body:

```xml
<body name="link7" ...>
    ...
    <site name="flange" pos="0 0 0"      quat="1 0 0 0" size="0.003" rgba="1 0 0 1"/>
    <site name="TCP"    pos="0 0 0.1034" quat="1 0 0 0" size="0.004" rgba="0 1 0 1"/>
</body>
```

Notes:

- If no tool is mounted, `TCP` can initially coincide with `flange`.
- If a gripper is mounted, keep `flange` at the robot mounting face and move `TCP` to the real tool center point.
- If a force/torque sensor site is used, a local name such as `FT` is appropriate.
- Keep these sites rigidly attached to the last robot link or tool body.

## 7. Default classes

`default class` names in the standalone model must also be local and unprefixed. Use names such as `robot`, `visual`, `collision`, `finger`, `size4`, and `joint1`. Do not define classes such as `Panda_robot`, `Panda_visual`, or `UR10e_size4` in the standalone model.

Example:

```xml
<default class="robot">
    <joint damping="1" armature="0.1"/>
    <default class="visual">
        <geom group="2"/>
    </default>
</default>
```

During scene generation, these class names are also prefixed automatically.

## 8. Minimal checklist for standalone robot MJCF files

- Base/root body: local name such as `base`
- Links: local names such as `link1`, `link7`, `base_link`, `wrist_3_link`
- Joints: local names such as `joint1 ... jointN` or `shoulder_pan_joint ... wrist_3_joint`
- Actuators: local names such as `actuator1 ... actuatorN`
- Joint sensors: `pos_jointi`, `vel_jointi`
- Flange site: `flange`
- TCP site: `TCP`
- Optional force/torque site: `FT`
- Cartesian sensors: `pos`, `ori`, `v`, `w`, `force`, `torque`
- Default classes: local names such as `robot`, `visual`, `collision`, `size4`, `joint1`

In short: keep the standalone robot MJCF file self-contained and unprefixed. Add prefixes only when generating the full scene.


# Imports


In [1]:
import numpy as np
from robotblockset.tools import get_rbs_path, print_xml

from robotblockset.transformations import rot_x, rot_y, rot_z, map_pose

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, replace_in_mjcf_file

# Initialization

Define the folder that contains the MJCF model files.


In [2]:
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"

Define the model names for the robot, gripper, camera, and tool that will be inserted into the scene.


In [3]:
ROBOT = "panda_no_hand"
ROBOT_NAME = "panda"
GRIPPER = "panda_hand"
CAMERA = "realsense_d435i"
TOOL = "calibration_plate"
PLATE = "charuco"

# Scene generation


## Basic scene

Load the base scene into which the robot, gripper, tool, and camera models will be inserted.


In [4]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "scene_high_resolution.xml")
scene.copy_during_attach = True

Use `print_xml` to inspect the current scene XML when debugging model composition.


In [5]:
print_xml(scene.to_xml())

## Robot


### Select robot


Load the robot MJCF specification and inspect its body tree with `print_body_tree`.


In [6]:
robot_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
robot_spec.copy_during_attach = True
print_body_tree(robot_spec.worldbody, robot_spec)


Body Tree for "world"
-base
--link0
---link1 (Joints: joint1-HINGE[Actuator: actuator1])
----link2 (Joints: joint2-HINGE[Actuator: actuator2])
-----link3 (Joints: joint3-HINGE[Actuator: actuator3])
------link4 (Joints: joint4-HINGE[Actuator: actuator4])
-------link5 (Joints: joint5-HINGE[Actuator: actuator5])
--------link6 (Joints: joint6-HINGE[Actuator: actuator6])
---------link7 (Joints: joint7-HINGE[Actuator: actuator7])
----------flange


Store the robot keyframe data so it can be reused when the final scene keyframes are rebuilt.


In [7]:
robot_ctrl = np.array(robot_spec.keys[0].ctrl)
robot_qpos = np.array(robot_spec.keys[0].qpos)

### Attach robot to ground

Define the world-frame attachment where the robot base will be inserted.


In [8]:
attachment_frame = scene.worldbody.add_frame(pos=[0, 0, 0], axisangle=[0, 0, 1, 0 * np.pi / 2])
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT_NAME}_")

Rename the generated robot base body. This is needed for automatic handle generation when creating the Python robot object.


In [9]:
scene.body(f"{ROBOT_NAME}_base").name = ROBOT_NAME

If the original robot MJCF file already contains a contact exclusion that references the original base body name, update that exclusion to use the renamed body.


In [10]:
# scene.excludes[i].bodyname1 = ROBOT_NAME

Optionally add contact exclusions between the robot and other scene objects.


In [11]:
# scene.add_exclude(bodyname1=f"{ROBOT_NAME}", bodyname2="ground")

## Gripper


### Attach gripper to robot


Load the gripper MJCF specification and store any keyframe data that must be merged into the final scene.


In [12]:
if GRIPPER is not None:
    gripper_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{GRIPPER}.xml")
    gripper_spec.copy_during_attach = True
    if len(gripper_spec.keys) > 0:
        gripper_ctrl = np.array(gripper_spec.keys[0].ctrl)
        gripper_qpos = np.array(gripper_spec.keys[0].qpos)
    else:
        gripper_ctrl = np.array([])
        gripper_qpos = np.array([])   


Define the gripper attachment frame, attach the gripper to the robot flange, and redefine the robot TCP. Inspect the resulting body tree to verify the attachment.


In [13]:
if GRIPPER is not None:
    attachment_frame = scene.body(f"{ROBOT_NAME}_flange").add_frame(pos=[0, 0, 0], quat=[0.9238795, 0, 0, -0.3826834])
    attachment_frame.attach_body(gripper_spec.body("hand"), f'{ROBOT_NAME}_tool_')
    scene.delete(scene.site(f"{ROBOT_NAME}_TCP"))
    print_body_tree(scene.worldbody, scene)



Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_actuator1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_actuator2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_actuator3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_actuator4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_actuator5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_actuator6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_actuator7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)


### Attach tool to gripper


Load the tool MJCF specification and inspect its body tree.


In [14]:
if TOOL is not None:
    tool_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{TOOL}.xml")
    tool_spec.copy_during_attach = True
    gripper_tcp_id = scene.site(f"{ROBOT_NAME}_tool_hand_TCP").id
    print_body_tree(tool_spec.worldbody, tool_spec)


Body Tree for "world"
-calib_charuco
-calib_checker


Define the attachment frame where the tool will be mounted on the gripper, then attach the tool model.


In [15]:
if TOOL is not None:
    scene.site(f"{ROBOT_NAME}_tool_hand_TCP").pos = scene.site(f"{ROBOT_NAME}_tool_hand_TCP").pos + [0, 0, tool_spec.geom(f"{PLATE}_pattern").size[1]]
    scene.site(f"{ROBOT_NAME}_tool_hand_TCP").quat = rot_y(np.pi / 2)    
    attachment_frame = scene.body(f"{ROBOT_NAME}_tool_hand").add_frame(pos=scene.site(f"{ROBOT_NAME}_tool_hand_TCP").pos, quat=scene.sites[gripper_tcp_id].quat)
    attachment_frame.attach_body(tool_spec.body(f"calib_{PLATE}"), "plate_")


Rename the tool TCP site to the robot TCP name so the robot wrapper can detect it automatically.


In [16]:
scene.site(f"{ROBOT_NAME}_tool_hand_TCP").name = f"{ROBOT_NAME}_TCP"


Inspect the final robot, gripper, and tool body tree.


In [17]:

print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_actuator1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_actuator2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_actuator3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_actuator4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_actuator5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_actuator6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_actuator7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)
------------plate_calib_charuco


## Cameras


Load the camera MJCF specification.


In [18]:
if CAMERA is not None:
    camera_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{CAMERA}.xml")
    camera_spec.copy_during_attach = True
    camera_spec.body("cam").mocap=0
    print_body_tree(camera_spec.worldbody, camera_spec)

Body Tree for "world"
-cam


Define the camera attachment frame in the world and attach the camera model.


In [19]:
if CAMERA is not None:
    attachment_frame = scene.worldbody.add_frame(pos=[1, 0, 0.6], axisangle=[0, 0, 1, 2 * np.pi / 2])
    attachment_frame.attach_body(camera_spec.body("cam"), "cam_")

## Prepare MJCF XML file


Rebuild the MJCF keyframes. Delete extra keys and define keyframe 0 as the initial configuration for the combined scene.


In [20]:
scene.keys

[]

In [21]:
_tmp = scene.to_xml()
while len(scene.keys)>1:
    scene.delete(scene.keys[-1])
for k in scene.keys:
    k.ctrl = np.concatenate([robot_ctrl, gripper_ctrl, np.zeros(len(scene.actuators) - len(robot_ctrl) - len(gripper_ctrl))])

Mark the camera body as `mocap` so it can be moved interactively in the application.


In [22]:
scene.body("cam_cam").mocap=1

Save the generated MJCF scene to an XML file.


In [23]:
scene.modelname = f"calibration_{PLATE}_scene"
scene.option.timestep = 0.001
with open(MODEL_PATH + f"{scene.modelname}.xml", "w") as f:
    f.write(scene.to_xml())

# Scene and robot definition verification

This section performs a minimal end-to-end check that the generated MJCF scene and the Python robot wrapper are consistent.

The verification does the following:

- loads the generated scene file `<ROBOT>_scene.xml` into `mujoco_scene`,
- creates the corresponding Python robot object on top of this scene,
- uses `JointNames="auto"` so the wrapper discovers the prefixed joint names directly from the generated scene,
- executes a simple `JMove(r.q_home)` command to verify that joints, actuators, and robot handles are connected correctly,
- prints the current Cartesian pose `r.x` to confirm that forward kinematics and TCP-related model definitions are valid.

In practice, this block verifies that scene generation added the expected prefixes and that the resulting model can be used immediately through the robot interface.


In [25]:
from robotblockset.mujoco.robots_pymujoco import *
scene = mujoco_scene(MODEL_PATH + f"{scene.modelname}.xml", show_camera=None)

In [28]:
r=eval(f"{ROBOT_NAME}(scene=scene, JointNames='auto')")

[RBS_INFO] [03:46:00.238] [panda_PyMuJoCo]: Robot connected to MuJoCo


In [29]:
r.JMove(r.q_home)
print(r.x)

[ 0.3047 -0.0001  0.3593  0.0013 -0.7114 -0.0016 -0.7028]


In [30]:
print(r.Kinmodel()[0])

[ 0.3047 -0.0001  0.3593  0.0013 -0.7114 -0.0016 -0.7028]
